In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import time
import numpy as np
import pandas as pd
import gc
from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBRegressor

PATH = '/kaggle/input/competitions/store-sales-time-series-forecasting/'

train  = pd.read_csv(PATH + 'train.csv',  parse_dates=['date'])
test   = pd.read_csv(PATH + 'test.csv',   parse_dates=['date'])
stores = pd.read_csv(PATH + 'stores.csv')
oil    = pd.read_csv(PATH + 'oil.csv',    parse_dates=['date'])
hol    = pd.read_csv(PATH + 'holidays_events.csv', parse_dates=['date'])

LAST_TRAIN_DAY = pd.Timestamp('2017-08-15')
TEST_START     = pd.Timestamp('2017-08-16')
T0 = time.time()


# 1. 完整数据网格 
df = pd.concat([train, test], ignore_index=True)

all_dates = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
store_ids = sorted(df['store_nbr'].unique())
fam_ids   = sorted(df['family'].unique())

grid = pd.MultiIndex.from_product([all_dates, store_ids, fam_ids],
                                  names=['date', 'store_nbr', 'family'])
df = df.set_index(['date', 'store_nbr', 'family']).reindex(grid).reset_index()

df.loc[df['sales'].isna() & (df['date'] < TEST_START), 'sales'] = 0
df['onpromotion'] = df['onpromotion'].fillna(0).astype('float32')


# 2. 静态特征 
df = df.merge(stores, on='store_nbr', how='left')

oil = pd.DataFrame({'date': all_dates}).merge(oil, on='date', how='left')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()
oil['oil_ma7'] = oil['dcoilwtico'].rolling(7, min_periods=1).mean()
df = df.merge(oil, on='date', how='left')

h = hol[(hol['locale'] == 'National') & (~hol['transferred']) & (hol['type'] != 'Work Day')]
h = h[['date']].drop_duplicates().assign(is_holiday=1)
df = df.merge(h, on='date', how='left')
df['is_holiday'] = df['is_holiday'].fillna(0).astype('int8')

df['year']       = df['date'].dt.year
df['month']      = df['date'].dt.month
df['day']        = df['date'].dt.day
df['dayofweek']  = df['date'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype('int8')
df['dayofyear']  = df['date'].dt.dayofyear
df['is_payday']  = ((df['day'] == 15) | df['date'].dt.is_month_end).astype('int8')
df['days_since_quake'] = (df['date'] - pd.Timestamp('2016-04-16')).dt.days

for col in ['city', 'state', 'type']:
    df[col + '_id'] = df[col].astype('category').cat.codes

df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

N_DATES  = len(all_dates)
N_SERIES = len(df) // N_DATES
assert N_SERIES * N_DATES == len(df), '网格不完整'
print(f'{N_SERIES} 条序列 × {N_DATES} 天 = {len(df)} 行')


# 3. 宽表 + 向量化特征
def to_wide(col):
    return pd.DataFrame(df[col].to_numpy('float32').reshape(N_SERIES, N_DATES).T)

def to_long(wide):
    return wide.to_numpy('float32').T.reshape(-1)

W  = to_wide('sales')
WP = to_wide('onpromotion')

df['lag_365']          = to_long(W.shift(365))
df['lag_364']          = to_long(W.shift(364))
df['last_year_mean15'] = to_long(W.shift(358).rolling(15, min_periods=1).mean())
df['promo_mean_14']    = to_long(WP.rolling(14, min_periods=1).mean())

for key, name in [('store_nbr', 'store'), ('family', 'fam')]:
    tot = df.groupby(['date', key])['sales'].sum().unstack()
    m28  = tot.shift(16).rolling(28,  min_periods=1).mean()
    m365 = tot.shift(16).rolling(365, min_periods=30).mean()
    ratio = m28 / (m365 + 1)
    df[f'{name}_mean28'] = df.set_index(['date', key]).index.map(m28.stack()).astype('float32')
    df[f'{name}_ratio']  = df.set_index(['date', key]).index.map(ratio.stack()).astype('float32')

del oil, hol, h, train, test
gc.collect()


# 4. 设置 
# 八个桶，每桶 2 天。(最短滞后天数, 第几天, 到第几天)
# 
BUCKETS = [(2, 1, 2), (4, 3, 4), (6, 5, 6), (8, 7, 8),
           (10, 9, 10), (12, 11, 12), (14, 13, 14), (16, 15, 16)]


VAL_STARTS = [pd.Timestamp(d) for d in ['2017-07-15', '2017-07-31']]
BASELINE = {'2017-07-15': 0.3846, '2017-07-31': 0.3905}   

FEATURES = [
    'store_nbr', 'city_id', 'state_id', 'type_id', 'cluster',
    'onpromotion', 'promo_mean_14',
    'year', 'month', 'day', 'dayofweek', 'is_weekend', 'dayofyear',
    'is_payday', 'is_holiday', 'days_since_quake', 'dcoilwtico', 'oil_ma7',
    'lag_365', 'lag_364', 'last_year_mean15',
    'store_mean28', 'store_ratio', 'fam_mean28', 'fam_ratio',
    'lag_a', 'lag_b', 'lag_c', 'lag_d',
    'rmean_7', 'rmean_14', 'rmean_28', 'rmean_60', 'rmean_140',
    'rstd_28', 'zero_60', 'dow_mean_8',
]

LGB_PARAMS = dict(objective='regression', learning_rate=0.05, num_leaves=63,
                  min_child_samples=20, colsample_bytree=0.8,
                  subsample=0.8, subsample_freq=1,
                  n_jobs=-1, random_state=42, verbose=-1)

XGB_PARAMS = dict(learning_rate=0.05, max_depth=6, min_child_weight=3,
                  subsample=0.8, colsample_bytree=0.8,
                  tree_method='hist', n_jobs=-1, random_state=42)


def build_bucket_features(H):
    sh = W.shift(H)
    df['lag_a'] = to_long(sh)
    df['lag_b'] = to_long(W.shift(H + 1))
    df['lag_c'] = to_long(W.shift(H + 2))
    df['lag_d'] = to_long(W.shift(H + 7))
    for w in [7, 14, 28, 60, 140]:
        df[f'rmean_{w}'] = to_long(sh.rolling(w, min_periods=1).mean())
    df['rstd_28'] = to_long(sh.rolling(28, min_periods=2).std())
    df['zero_60'] = to_long((sh == 0).rolling(60, min_periods=1).mean())
    k0 = int(np.ceil(H / 7) * 7)
    df['dow_mean_8'] = to_long(sum(W.shift(k0 + 7 * i) for i in range(8)) / 8)


def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(np.clip(y_pred, 0, None)) - np.log1p(y_true)) ** 2))


# 5. 训练 
in_range = df['date'] >= '2015-01-01'
val_parts, test_parts = [], []

for H, lo, hi in BUCKETS:
    t_bucket = time.time()
    print(f'\n===== 桶：第 {lo}-{hi} 天，最短用 {H} 天前的数据 =====')
    build_bucket_features(H)
    rounds_lgb = {f: [] for f in fam_ids}
    rounds_xgb = {f: [] for f in fam_ids}

    for w_start in VAL_STARTS:
        L = w_start - pd.Timedelta(days=1)
        fit_m = in_range & (df['date'] < w_start)
        val_m = (df['date'] >= L + pd.Timedelta(days=lo)) & \
                (df['date'] <= L + pd.Timedelta(days=hi))

        val_df = df.loc[val_m, ['family', 'sales']].copy()
        for c in ['pred', 'pred_lgb', 'pred_xgb']:
            val_df[c] = np.nan

        for fam in fam_ids:
            a = df[fit_m & (df['family'] == fam)]
            b = df[val_m & (df['family'] == fam)]
            ya, yb = np.log1p(a['sales']), np.log1p(b['sales'])

            ml = LGBMRegressor(n_estimators=1200, **LGB_PARAMS)
            ml.fit(a[FEATURES], ya, eval_set=[(b[FEATURES], yb)], eval_metric='rmse',
                   callbacks=[early_stopping(50, verbose=False)])
            pl = ml.predict(b[FEATURES])
            rounds_lgb[fam].append(max(ml.best_iteration_ or 50, 30))

            mx = XGBRegressor(n_estimators=1200, early_stopping_rounds=50,
                              eval_metric='rmse', **XGB_PARAMS)
            mx.fit(a[FEATURES], ya, eval_set=[(b[FEATURES], yb)], verbose=False)
            px = mx.predict(b[FEATURES])
            rounds_xgb[fam].append(max(mx.best_iteration, 30) + 1)

            # 融合必须在对数空间做。指标是 RMSLE。

            val_df.loc[b.index, 'pred_lgb'] = np.expm1(pl).clip(0)
            val_df.loc[b.index, 'pred_xgb'] = np.expm1(px).clip(0)
            val_df.loc[b.index, 'pred']     = np.expm1((pl + px) / 2).clip(0)

        assert val_df['pred'].notna().all(), '有验证行没被填到'
        val_df['window'] = w_start
        val_parts.append(val_df)
        print(f'  窗口 {w_start.date()}  本桶 RMSLE = '
              f'{rmsle(val_df["sales"].values, val_df["pred"].values):.4f}')

    # 全量重训
    fit_m  = in_range & (df['date'] <= LAST_TRAIN_DAY)
    test_m = (df['date'] >= LAST_TRAIN_DAY + pd.Timedelta(days=lo)) & \
             (df['date'] <= LAST_TRAIN_DAY + pd.Timedelta(days=hi))

    out = df.loc[test_m, ['id', 'rmean_140']].copy()
    out['pred'] = np.nan
    for fam in fam_ids:
        a = df[fit_m & (df['family'] == fam)]
        b = df[test_m & (df['family'] == fam)]
        ya = np.log1p(a['sales'])

        ml = LGBMRegressor(n_estimators=int(np.mean(rounds_lgb[fam])), **LGB_PARAMS)
        ml.fit(a[FEATURES], ya)
        mx = XGBRegressor(n_estimators=int(np.mean(rounds_xgb[fam])), **XGB_PARAMS)
        mx.fit(a[FEATURES], ya)

        pl = ml.predict(b[FEATURES])
        px = mx.predict(b[FEATURES])
        out.loc[b.index, 'pred'] = np.expm1((pl + px) / 2).clip(0)

    assert out['pred'].notna().all(), '有测试行没被填到'
    test_parts.append(out)
    print(f'  本桶用时 {(time.time() - t_bucket) / 60:.1f} 分钟，'
          f'累计 {(time.time() - T0) / 60:.1f} 分钟')
    gc.collect()


# 6. 验证结果 
allv = pd.concat(val_parts, ignore_index=True)

print('\n========== 每个验证窗口的整体分数（八个桶合起来）==========')
scores = []
for w_start, grp in allv.groupby('window'):
    s = rmsle(grp['sales'].values, grp['pred'].values)
    base = BASELINE[str(w_start.date())]
    scores.append(s)
    print(f'  {w_start.date()}  融合 {s:.4f}   v5.1 {base:.4f}   差值 {s - base:+.4f}')
print(f'\n融合平均 = {np.mean(scores):.4f}')

# 单模型分数
for c, tag in [('pred_lgb', 'LightGBM 单独'), ('pred_xgb', 'XGBoost 单独')]:
    s = [rmsle(g['sales'].values, g[c].values) for _, g in allv.groupby('window')]
    print(f'{tag}平均 = {np.mean(s):.4f}')

allv['err2'] = (np.log1p(allv['pred']) - np.log1p(allv['sales'])) ** 2
print('\n========== 分品类误差（误差最大的 10 个）==========')
print(allv.groupby('family')
          .agg(rmsle=('err2', lambda x: np.sqrt(x.mean())),
               avg_sales=('sales', 'mean'), avg_pred=('pred', 'mean'))
          .sort_values('rmsle', ascending=False).head(10).round(3))


# 7. 提交文件 
sub = pd.concat(test_parts, ignore_index=True)
sub['sales'] = np.where(sub['rmean_140'].values == 0, 0, sub['pred'].values)
sub = sub[['id', 'sales']].copy()
sub['id'] = sub['id'].astype(int)
sub = sub.sort_values('id').reset_index(drop=True)
sub.to_csv('submission.csv', index=False)

print('\n', sub.shape, '应该是 (28512, 2)')
print(sub.head())
print(f'总用时 {(time.time() - T0) / 60:.1f} 分钟')

1782 条序列 × 1704 天 = 3036528 行

===== 桶：第 1-2 天，最短用 2 天前的数据 =====
  窗口 2017-07-15  本桶 RMSLE = 0.3474
  窗口 2017-07-31  本桶 RMSLE = 0.3745
  本桶用时 6.3 分钟，累计 6.6 分钟

===== 桶：第 3-4 天，最短用 4 天前的数据 =====
  窗口 2017-07-15  本桶 RMSLE = 0.3638
  窗口 2017-07-31  本桶 RMSLE = 0.3770
  本桶用时 5.8 分钟，累计 12.4 分钟

===== 桶：第 5-6 天，最短用 6 天前的数据 =====
  窗口 2017-07-15  本桶 RMSLE = 0.3844
  窗口 2017-07-31  本桶 RMSLE = 0.3708
  本桶用时 5.3 分钟，累计 17.7 分钟

===== 桶：第 7-8 天，最短用 8 天前的数据 =====
  窗口 2017-07-15  本桶 RMSLE = 0.3750
  窗口 2017-07-31  本桶 RMSLE = 0.3811
  本桶用时 6.2 分钟，累计 23.9 分钟

===== 桶：第 9-10 天，最短用 10 天前的数据 =====
  窗口 2017-07-15  本桶 RMSLE = 0.3899
  窗口 2017-07-31  本桶 RMSLE = 0.3866
  本桶用时 5.3 分钟，累计 29.2 分钟

===== 桶：第 11-12 天，最短用 12 天前的数据 =====
  窗口 2017-07-15  本桶 RMSLE = 0.3988
  窗口 2017-07-31  本桶 RMSLE = 0.4051
  本桶用时 6.8 分钟，累计 36.0 分钟

===== 桶：第 13-14 天，最短用 14 天前的数据 =====
  窗口 2017-07-15  本桶 RMSLE = 0.4025
  窗口 2017-07-31  本桶 RMSLE = 0.3904
  本桶用时 4.9 分钟，累计 40.8 分钟

===== 桶：第 15-16 天，最短用 16 天前的数据 =====
  窗口 2017-07-15